# 01 - Data Acquisition

**Purpose:** Query the Materials Project API for elasticity records and filter the results down to binary and ternary transition-metal intermetallic compounds. The cleaned raw dataset is written to `../data/bradley_intermetallics_elasticity.csv` for use by the downstream notebooks.

**Inputs:** Materials Project API key (`MP_API_KEY`), loaded from a local `.env` file (never committed to version control).

**Output:** `data/bradley_intermetallics_elasticity.csv`

## Authenticate and query the Materials Project elasticity endpoint

In [1]:
# Load the Materials Project API key from a local .env file rather than hardcoding it,
# so the notebook can be shared/committed without leaking credentials.
import os
from dotenv import load_dotenv
from mp_api.client import MPRester

load_dotenv()  # walks up from the notebook's directory to find the repo-root .env file
api_key = os.getenv("MP_API_KEY")

if not api_key:
    raise ValueError(
        "MP_API_KEY not found. Copy .env.example to .env at the repo root and add your key."
    )

print("Authenticated via .env. Querying the Materials Project elasticity endpoint...")

with MPRester(api_key) as mpr:
    elasticity_docs = mpr.materials.elasticity.search(
        fields=["material_id", "formula_pretty", "bulk_modulus", "symmetry", "elements"]
    )

print(f"Server returned {len(elasticity_docs)} total elasticity records.")

Authenticated via .env. Querying the Materials Project elasticity endpoint...


Retrieving ElasticityDoc documents:   0%|          | 0/13283 [00:00<?, ?it/s]

Server returned 13283 total elasticity records.


## Filter to transition-metal binary/ternary intermetallics and clean the target

In [2]:
import re
import pandas as pd

# Transition metals we consider valid intermetallic-forming elements
transition_metals = {
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "La", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg",
}

# Non-metals to exclude so the remaining compounds are true intermetallics (metal-metal only)
excluded_elements = {"O", "N", "S", "Se", "Te", "F", "Cl", "Br", "I", "H", "C", "P", "As"}

# Sanity bounds (GPa) for bulk modulus: a small number of MP elasticity entries come from
# DFT calculations that did not fully converge and report unphysical values, so these are
# dropped here as part of the missing/invalid-data cleaning step.
BULK_MODULUS_MIN_GPA, BULK_MODULUS_MAX_GPA = 0, 1000

records = []
n_dropped_unphysical = 0
for doc in elasticity_docs:
    try:
        formula = getattr(doc, "formula_pretty", None)
        if not formula:
            continue

        elements_obj = getattr(doc, "elements", None)
        if elements_obj:
            elements = {el.symbol if hasattr(el, "symbol") else str(el) for el in elements_obj}
        else:
            elements = set(re.findall(r'[A-Z][a-z]?', formula))

        num_elements = len(elements)

        # Keep binary and ternary compounds only
        if num_elements not in [2, 3]:
            continue

        # Exclude non-metals to guarantee pure intermetallic phases
        if elements.intersection(excluded_elements):
            continue

        # Require at least one transition metal
        if not elements.intersection(transition_metals):
            continue

        # Extract bulk modulus VRH (handles both dict and string representations returned by the API)
        bulk_modulus = getattr(doc, "bulk_modulus", None)
        if bulk_modulus is None:
            continue

        if isinstance(bulk_modulus, dict):
            bulk_modulus_vrh = bulk_modulus.get("vrh", None) or bulk_modulus.get("homogeneous", None)
        else:
            bulk_modulus_str = str(bulk_modulus)
            match = re.search(r'vrh=([0-9.]+)', bulk_modulus_str)
            bulk_modulus_vrh = float(match.group(1)) if match else float(bulk_modulus)

        if bulk_modulus_vrh is None:
            continue

        if not (BULK_MODULUS_MIN_GPA < bulk_modulus_vrh < BULK_MODULUS_MAX_GPA):
            n_dropped_unphysical += 1
            continue

        symmetry_obj = getattr(doc, "symmetry", None)
        if hasattr(symmetry_obj, "crystal_system"):
            crystal_system = getattr(symmetry_obj.crystal_system, "name", str(symmetry_obj.crystal_system)).lower()
        else:
            crystal_system = str(symmetry_obj).lower()

        records.append({
            "material_id": str(getattr(doc, "material_id")),
            "formula": formula,
            "bulk_modulus_vrh": float(bulk_modulus_vrh),
            "crystal_system": crystal_system,
            "elements": sorted(elements),
            "num_elements": num_elements,
        })

    except Exception:
        continue

intermetallics_df = pd.DataFrame(records)
intermetallics_df = intermetallics_df.dropna(subset=["bulk_modulus_vrh", "crystal_system"]).drop_duplicates(subset=["material_id"])

# Save with a relative path so this works regardless of where the repo is cloned
intermetallics_df.to_csv("../data/bradley_intermetallics_elasticity.csv", index=False)

print(f"Dropped {n_dropped_unphysical} records with unphysical bulk modulus (outside {BULK_MODULUS_MIN_GPA}-{BULK_MODULUS_MAX_GPA} GPa).")
print(f"\nFinal dataset shape: {intermetallics_df.shape}")
print("\nBulk Modulus VRH (GPa) summary statistics:")
print(intermetallics_df["bulk_modulus_vrh"].describe())

Dropped 32 records with unphysical bulk modulus (outside 0-1000 GPa).

Final dataset shape: (5242, 6)

Bulk Modulus VRH (GPa) summary statistics:
count    5242.000000
mean      127.079249
std        71.986020
min         3.088000
25%        72.706500
50%       113.916000
75%       171.526500
max       873.978000
Name: bulk_modulus_vrh, dtype: float64
